# ACTIVIDAD SESIÓN 5 (RESUELTA): INTRODUCCIÓN A MACHINE LEARNING ESCALABLE

Una tienda de cosmética quiere desarrollar un sistema inteligente que clasifique productos de *skin care* en diferentes categorías según sus características.

**Objetivo**  
Entrenar un modelo de clasificación con **MLlib** que prediga el tipo de piel recomendado para cada producto.

**Dataset**: `skincare_products.csv`

## 1. Carga y exploración de datos (2 puntos)

In [2]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = SparkSession.builder.appName("SkinCareML").getOrCreate()

# Intentar cargar dataset
try:
    df = spark.read.csv("skincare_products.csv", header=True, inferSchema=True)
    origen = "archivo skincare_products.csv"
except Exception:
    data = [
        ("Ácido Hialurónico", "Alto", "Alto", 30, "Seco"),
        ("Retinol", "Medio", "Alto", 0, "Graso"),
        ("Vitamina C", "Medio", "Medio", 15, "Mixto"),
        ("Ceramidas", "Alto", "Medio", 20, "Sensible"),
        ("Niacinamida", "Bajo", "Alto", 0, "Graso"),
    ]
    schema = ["ingredientes", "hidratacion", "absorcion", "spf", "tipo_piel"]
    df = spark.createDataFrame(data, schema)
    origen = "datos embebidos (fallback)"

print(f"Origen de datos: {origen}")
df.show(5)
df.describe().show()

Origen de datos: archivo skincare_products.csv
+-----------------+-----------+---------+---+------------+
|     Ingredientes|Hidratación|Absorción|SPF|Tipo de Piel|
+-----------------+-----------+---------+---+------------+
|Ácido Hialurónico|       Alto|    Medio|  0|        Seco|
|          Retinol|       Bajo|     Alto|  0|       Graso|
|       Vitamina C|      Medio|    Medio| 30|       Mixto|
|        Aloe Vera|       Alto|     Bajo| 15|    Sensible|
|      Niacinamida|      Medio|    Medio|  0|       Mixto|
+-----------------+-----------+---------+---+------------+
only showing top 5 rows
+-------+----------------+-----------+---------+-----------------+------------+
|summary|    Ingredientes|Hidratación|Absorción|              SPF|Tipo de Piel|
+-------+----------------+-----------+---------+-----------------+------------+
|  count|              20|         20|       20|               20|          20|
|   mean|            NULL|       NULL|     NULL|              7.5|        NULL

## 2. Preprocesamiento de datos (2 puntos)

In [3]:
from pyspark.ml.feature import VectorAssembler

# Mapear Tipo de Piel manualmente
df = df.withColumn("label",
    F.when(F.col("tipo_piel")=="Seco", 0)
     .when(F.col("tipo_piel")=="Graso", 1)
     .when(F.col("tipo_piel")=="Mixto", 2)
     .when(F.col("tipo_piel")=="Sensible", 3)
)

# Mapear Hidratación y Absorción
map_hid = {"Bajo":0, "Medio":1, "Alto":2}
map_abs = {"Bajo":0, "Medio":1, "Alto":2}

df = df.replace(map_hid, subset=["hidratacion"])       .replace(map_abs, subset=["absorcion"])       .withColumnRenamed("hidratacion", "hidratacion_num")       .withColumnRenamed("absorcion", "absorcion_num")

# Ensamblar features
assembler = VectorAssembler(
    inputCols=["hidratacion_num", "absorcion_num", "spf"],
    outputCol="features"
)
df_ready = assembler.transform(df)
df_ready.select("ingredientes","features","label").show(truncate=False)

{"ts": "2025-09-08 21:59:04.602", "level": "ERROR", "logger": "DataFrameQueryContextLogger", "msg": "[UNRESOLVED_COLUMN.WITH_SUGGESTION] A column, variable, or function parameter with name `tipo_piel` cannot be resolved. Did you mean one of the following? [`Tipo de Piel`, `Absorción`, `SPF`, `Hidratación`, `Ingredientes`]. SQLSTATE: 42703", "context": {"file": "line 5 in cell [3]", "line": "", "fragment": "col", "errorClass": "UNRESOLVED_COLUMN.WITH_SUGGESTION"}, "exception": {"class": "Py4JJavaError", "msg": "An error occurred while calling o44.withColumn.\n: org.apache.spark.sql.AnalysisException: [UNRESOLVED_COLUMN.WITH_SUGGESTION] A column, variable, or function parameter with name `tipo_piel` cannot be resolved. Did you mean one of the following? [`Tipo de Piel`, `Absorción`, `SPF`, `Hidratación`, `Ingredientes`]. SQLSTATE: 42703;\n'Project [Ingredientes#405, Hidratación#406, Absorción#407, SPF#408, Tipo de Piel#409, CASE WHEN '`=`('tipo_piel, Seco) THEN 0 WHEN '`=`('tipo_piel, Gr

AnalysisException: [UNRESOLVED_COLUMN.WITH_SUGGESTION] A column, variable, or function parameter with name `tipo_piel` cannot be resolved. Did you mean one of the following? [`Tipo de Piel`, `Absorción`, `SPF`, `Hidratación`, `Ingredientes`]. SQLSTATE: 42703;
'Project [Ingredientes#405, Hidratación#406, Absorción#407, SPF#408, Tipo de Piel#409, CASE WHEN '`=`('tipo_piel, Seco) THEN 0 WHEN '`=`('tipo_piel, Graso) THEN 1 WHEN '`=`('tipo_piel, Mixto) THEN 2 WHEN '`=`('tipo_piel, Sensible) THEN 3 END AS label#776]
+- Relation [Ingredientes#405,Hidratación#406,Absorción#407,SPF#408,Tipo de Piel#409] csv


## 3. División de datos y entrenamiento del modelo (3 puntos)

In [ ]:
from pyspark.ml.classification import DecisionTreeClassifier

train, test = df_ready.randomSplit([0.8,0.2], seed=42)

dt = DecisionTreeClassifier(featuresCol="features", labelCol="label")
model = dt.fit(train)

print("Modelo entrenado:")
print(model.toDebugString)

## 4. Predicción y evaluación (2 puntos)

In [ ]:
predictions = model.transform(test)
predictions.select("ingredientes","features","label","prediction").show()

from pyspark.ml.evaluation import MulticlassClassificationEvaluator

evaluator = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="accuracy")
accuracy = evaluator.evaluate(predictions)
print(f"Precisión del modelo: {accuracy:.2f}")

## 5. Análisis de resultados y mejoras (1 punto)

El modelo de Árbol de Decisión obtuvo una precisión estimada según el dataset de prueba.  
Posibles mejoras:
- Probar otros algoritmos como **Random Forest** o **Gradient Boosted Trees**.  
- Ajustar hiperparámetros (profundidad del árbol, criterio de división).  
- Usar un dataset más grande y balanceado para mejorar la generalización.

## (Opcional) Detener Spark

In [ ]:
spark.stop()